## Notes from work with Abhi regarding conversion of Geoid referenced data (NAVD88; EPSG 5703) to ellipsoid referenced data (WGS 84/ UTM 11N; EPSG 32611):

I ended up downloading the AWS data in EPSG 6340 (NAD83/ UTM 11N). So 6340 height, NAVD88 (EPSG 5703) heights, which is standard for 3DEP. 
Used a JSON to convert NAD83/UTM to geographic coordinates AND perform that calculation to get from NAVD88 to NAD83 (. I initially used a .gtx grid file but PROJ wasn't happy with that, so I downloaded a .tif from here instead- https://cdn.proj.org/. Resources for this process can be found here:https://proj.org/en/stable/usage/transformation.html. From this website, an example of this transformation is provided:

E.g. when going from UTM zone 32, datum ED50, to UTM zone 32, datum ETRS89, one must, in the simplest case, go through 5 steps:

1. Back-project the UTM coordinates to geographic coordinates
2. Convert the geographic coordinates to 3D cartesian geocentric coordinates
3. Apply a Helmert transformation from ED50 to ETRS89
4. Convert back from cartesian to geographic coordinates
5. Finally project the geographic coordinates to UTM zone 32 planar coordinates.



Use `projinfo` to find PROJ strings needed for conversions between CRS. `-s` indicated source CRS, `-t` indicates target CRS, `-o` indicated output format (that being `PROJ`), `q` indicates Quiet Mode, meaning it suppresses additional output and only returns the essential transformation information:

In [7]:
## NAD83 (2011)/ UTM 11 N --> NAD83 (2011) [Geographic coordinates]
!projinfo -s "EPSG:6340" -t "EPSG:6319" -o PROJ -q  

+proj=pipeline
  +step +inv +proj=utm +zone=11 +ellps=GRS80
  +step +proj=unitconvert +xy_in=rad +z_in=m +xy_out=deg +z_out=m
  +step +proj=axisswap +order=2,1


2. Apply geoid 12b grid shift to geographic coordinates:

+proj=pipeline +step +proj=axisswap +order=2,1 +step +proj=unitconvert +xy_in=deg +xy_out=rad +step +proj=vgridshift +grids="g2018u0.bin" +multiplier=1 +step +proj=unitconvert +xy_in=rad +xy_out=deg +step +proj=axisswap +order=2,1

In [10]:
!projinfo -s "EPSG:5703" -t "EPSG:6340" -o PROJ -q  


+proj=noop


In [ ]:
input_laz = "path to laz file"
geoid_grid = "path to geoid grid file"

In [9]:
import json
import pdal

# Define the PDAL pipeline JSON structure
pipeline_json = [
	{
	"type" : "readers.las",
	"filename" : "pc_sample.laz"
	},

	{
	"type":"filters.projpipeline",
	"coord_op":"+proj=pipeline +step +inv +proj=utm +zone=11 +ellps=GRS80 +step +proj=unitconvert +xy_in=rad +xy_out=deg +step +proj=axisswap +order=2,1"
	},
               #This step inverts a UTM (Universal Transverse Mercator) projection transformation.
					#+zone=11 specifies UTM Zone 11.
					#+ellps=GRS80 sets the GRS80 ellipsoid.
    				# Since it is inverted, it converts from UTM coordinates back to geographic (latitude/longitude).
    				#6340 -> 4269
    
	{
	"type":"filters.projpipeline",	
	"coord_op":"+proj=pipeline +step +proj=axisswap +order=2,1 +step +proj=unitconvert +xy_in=deg +xy_out=rad +step +proj=vgridshift +grids=us_noaa_geoid09_conus.tif +multiplier=1 +step +proj=unitconvert +xy_in=rad +xy_out=deg +step +proj=axisswap +order=2,1"
	},
	{
	"type":"filters.projpipeline",	
	"coord_op":"+proj=pipeline +step +proj=axisswap +order=2,1 +step +proj=unitconvert +xy_in=deg +xy_out=rad +step +proj=utm +zone=11 +ellps=GRS80"
	},
	{
	"type" : "writers.las",
	"filename" : "pc_ellipsoid.laz",
	"forward": "all",
	"minor_version":"2",
	"a_srs": "EPSG:6340",
	"system_id":"RIEGL-580-II",
	"software_id":"TERRASCAN_PDAL"
	}
]

# Convert pipeline to JSON string
pipeline_str = json.dumps(pipeline_json, indent=4)

# Run the PDAL pipeline
pipeline = pdal.Pipeline(pipeline_str)
try:
    pipeline.execute()
    print("PDAL pipeline executed successfully.")
except RuntimeError as e:
    print(f"Error executing PDAL pipeline: {e}")


SystemError: <built-in function isinstance> returned a result with an error set

3. Used a second json to convert from NAD83 to WGS84 AND WGS84 geographic coordinated --> WGS84/UTM11N:


In [ ]:
# Define the PDAL pipeline JSON structure
pipeline_json = [
[
	{
	"type" : "readers.las",
	"filename" : "pc_ellipsoid.laz"
	},
	{
	"type": "filters.projpipeline",
	"coord_op": "  +proj=pipeline +step +inv +proj=utm +zone=11 +ellps=GRS80 +step +proj=cart +ellps=GRS80 +step +inv +proj=helmert +x=1.0053 +y=-1.90921 +z=-0.54157 +rx=0.02678138 +ry=-0.00042027 +rz=0.01093206 +s=0.00036891 +dx=0.00079 +dy=-0.0006 +dz=-0.00144 +drx=6.667e-05 +dry=-0.00075744 +drz=-5.133e-05 +ds=-7.201e-05 +t_epoch=2010 +convention=coordinate_frame +step +inv +proj=cart +ellps=GRS80 +step +proj=unitconvert +xy_in=rad +z_in=m +xy_out=deg +z_out=m +step +proj=axisswap +order=2,1"
	},
	{
	"type": "filters.projpipeline",
	"coord_op":"+proj=pipeline +step +proj=axisswap +order=2,1 +step +proj=unitconvert +xy_in=deg +xy_out=rad +step +proj=utm +zone=11 +ellps=WGS84"
	},
	{
	"type" : "writers.las",
	"filename" : "pc_WGS84.laz",
	"forward": "all",
	"minor_version":"2",
	"a_srs": "EPSG:32611",
	"system_id":"RIEGL-580-II"
	}
]]
    
# Convert pipeline to JSON string
pipeline_str = json.dumps(pipeline_json, indent=4)

# Run the PDAL pipeline
pipeline = pdal.Pipeline(pipeline_str)
try:
    pipeline.execute()
    print("PDAL pipeline executed successfully.")
except RuntimeError as e:
    print(f"Error executing PDAL pipeline: {e}")
